## Step 2.1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Verify the mount

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/nbe_rag_pipeline"
IMAGES_DIR = os.path.join(BASE_DIR, "docs_images")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Verify docs_images subfolders exist
for doc_id in ["rtgs", "legal_circular_1", "legal_circular_2"]:
    path = os.path.join(IMAGES_DIR, doc_id)
    count = len([f for f in os.listdir(path) if f.endswith(".png")])
    print(f"  {doc_id}: {count} pages found")


  rtgs: 19 pages found
  legal_circular_1: 2 pages found
  legal_circular_2: 31 pages found


## Step 2.2 — Install Dependencies

In [ ]:
# Install Chandra OCR with Hugging Face backend, and bitsandbytes for quantization
!pip install chandra-ocr[hf] bitsandbytes>=0.46.1 --quiet

# Install Pillow (for image loading)
!pip install Pillow --quiet

### Verify the install

In [ ]:
import chandra
import importlib.metadata

print(f"chandra-ocr version: {importlib.metadata.version('chandra-ocr')}")

chandra-ocr version: 0.2.0


## Step 2.3 — Authenticate with Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to Hugging Face ✓")

Logged in to Hugging Face ✓


## Step 2.4 — Load the Chandra OCR-2 Model

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("Memory allocator config set ✓")

Memory allocator config set ✓


### Load the model quantized into 4 bits

In [ ]:
import torch
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)

# 4-bit quantization config
# Reduces model VRAM from ~8 GB → ~2 GB on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,   # compute in bfloat16 for quality
    bnb_4bit_quant_type="nf4",               # NF4 = best accuracy for 4-bit
    bnb_4bit_use_double_quant=True,          # nested quantization, saves extra ~0.4 GB
)

print("Loading Chandra OCR-2 in 4-bit (NF4) quantization...")

model = AutoModelForImageTextToText.from_pretrained(
    "datalab-to/chandra-ocr-2",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
model.processor = AutoProcessor.from_pretrained(
    "datalab-to/chandra-ocr-2",
    token=HF_TOKEN,
)
model.processor.tokenizer.padding_side = "left"

# Confirm memory usage
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
free      = torch.cuda.mem_get_info()[0] / 1e9
print(f"\nModel loaded ✓")
print(f"  Allocated : {allocated:.2f} GB")
print(f"  Reserved  : {reserved:.2f} GB")
print(f"  Free      : {free:.2f} GB")
print(f"\nExpected: allocated ~2.0 GB, free ~11+ GB")


Loading Chandra OCR-2 in 4-bit (NF4) quantization...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]


Model loaded ✓
  Allocated : 3.29 GB
  Reserved  : 5.24 GB
  Free      : 10.27 GB

Expected: allocated ~2.0 GB, free ~11+ GB


### Loading the full model requires about 12 GB VRAM

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown

print("Loading Chandra OCR-2 model...")

model = AutoModelForImageTextToText.from_pretrained(
    "datalab-to/chandra-ocr-2",
    dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
model.processor = AutoProcessor.from_pretrained(
    "datalab-to/chandra-ocr-2",
    token=HF_TOKEN,
)
model.processor.tokenizer.padding_side = "left"

# Confirm GPU usage
device = next(model.parameters()).device
print(f"Model loaded on: {device}")
print("Model ready ✓")

Loading Chandra OCR-2 model...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

Model loaded on: cuda:0
Model ready ✓


## Step 2.5 — Define Document Registry

In [ ]:
DOCUMENT_REGISTRY = {
    "rtgs": {
        "doc_id": "rtgs",
        "doc_name": "egyptian_multicurrency_rtgs_procedures.pdf",
        "doc_type": "SOP",
        "language": "ar",
        "total_pages": 19,
    },
    "legal_circular_1": {
        "doc_id": "legal_circular_1",
        "doc_name": "legal_circular_1.pdf",
        "doc_type": "Legal Circular",
        "language": "ar",
        "total_pages": 2,
    },
    "legal_circular_2": {
        "doc_id": "legal_circular_2",
        "doc_name": "legal_circular_2.pdf",
        "doc_type": "Legal Circular",
        "language": "ar",
        "total_pages": 31,
    },
}

## Step 2.6 — Define the Per-Page Extraction Function

In [ ]:
import torch, gc, json, re
from PIL import Image
from datetime import datetime
from bs4 import BeautifulSoup   # pre-installed in Colab
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem

MAX_INFERENCE_DIM = 1400

# Maps data-label values from the model HTML to our schema block_type values
LABEL_MAP = {
    "page-header":   "header",
    "page-footer":   "footer",
    "text":          "text",
    "table":         "table",
    "figure":        "figure",
    "list":          "list",
    "section-header":"header",
    "caption":       "text",
}


def resize_for_inference(
    image: Image.Image,
    max_dim: int = MAX_INFERENCE_DIM,
) -> tuple[Image.Image, float]:
    w, h = image.size
    if max(w, h) <= max_dim:
        return image, 1.0
    scale = max_dim / max(w, h)
    return image.resize((int(w * scale), int(h * scale)), Image.LANCZOS), scale


def parse_blocks_from_html(
    raw_html: str,
    doc_id: str,
    page_num: int,
    inf_w: int,
    inf_h: int,
    scale: float,
    default_language: str,
) -> list:
    """
    Parse result.raw (HTML string) into our block schema.

    The model emits HTML like:
      <div data-bbox="113 315 886 390" data-label="Text">
        <p>في إطار حرص البنك المركزي...</p>
      </div>

    bbox values are pixel coordinates in the inference image.
    We scale them back to original image coordinates.
    """
    soup = BeautifulSoup(raw_html, "html.parser")
    blocks = []
    inv = 1.0 / scale  # multiplier to go from inference → original pixels

    for idx, div in enumerate(soup.find_all("div", attrs={"data-bbox": True})):
        block_id = f"{doc_id}_p{page_num:03d}_b{idx:02d}"

        # --- Block type ---
        raw_label = div.get("data-label", "text").lower().strip()
        block_type = LABEL_MAP.get(raw_label, "text")

        # --- Bounding box ---
        bbox = None
        bbox_str = div.get("data-bbox", "").strip()
        if bbox_str:
            try:
                parts = list(map(float, bbox_str.split()))
                if len(parts) == 4:
                    x1, y1, x2, y2 = parts
                    # Scale from inference-image coords → original image coords
                    bbox = [
                        round(x1 * inv),
                        round(y1 * inv),
                        round(x2 * inv),
                        round(y2 * inv),
                    ]
            except (ValueError, TypeError):
                bbox = None

        # --- Text content ---
        text = div.get_text(separator=" ", strip=True)

        # --- Table data (keep raw HTML for table blocks for structured parsing later) ---
        table_data = None
        if block_type == "table":
            table_html = div.decode_contents()  # inner HTML of the table div
            table_data = table_html.strip()

        blocks.append({
            "block_id":   block_id,
            "block_type": block_type,
            "text":       text,
            "bbox":       bbox,
            "confidence": None,   # model does not emit confidence scores
            "language":   default_language,
            "table_data": table_data,
            "crop_path":  None,   # filled in Stage 3
        })

    return blocks


def extract_page(doc_id: str, page_num: int, image_path: str) -> dict:
    """
    Run Chandra OCR-2 on one page image and return a structured JSON dict.
    All bounding boxes are in original image pixel coordinates.
    """
    meta = DOCUMENT_REGISTRY[doc_id]

    # Load + resize
    original_image = Image.open(image_path).convert("RGB")
    orig_w, orig_h = original_image.size
    inference_image, scale = resize_for_inference(original_image)
    inf_w, inf_h = inference_image.size
    print(f"    [{orig_w}×{orig_h} → {inf_w}×{inf_h}, scale={scale:.3f}]", end=" ", flush=True)

    batch = [BatchInputItem(image=inference_image, prompt_type="ocr_layout")]

    with torch.inference_mode():
        result = generate_hf(batch, model)[0]

    # Parse the HTML output into structured blocks
    blocks = parse_blocks_from_html(
        raw_html=result.raw,
        doc_id=doc_id,
        page_num=page_num,
        inf_w=inf_w,
        inf_h=inf_h,
        scale=scale,
        default_language=meta["language"],
    )

    # Cleanup GPU memory
    del inference_image, batch, result, original_image
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "doc_id":               doc_id,
        "doc_name":             meta["doc_name"],
        "doc_type":             meta["doc_type"],
        "page_num":             page_num,
        "original_size":        [orig_w, orig_h],
        "page_image_path":      f"docs_images/{doc_id}/page_{page_num:03d}.png",
        "extraction_timestamp": datetime.utcnow().isoformat() + "Z",
        "blocks":               blocks,
    }


### Inspect the object returned by the model

In [ ]:
import torch
from PIL import Image
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem

# Diagnostic: Check the structure of the model output
test_image_path = os.path.join(IMAGES_DIR, "rtgs", "page_001.png")
original_image = Image.open(test_image_path).convert("RGB")

# Resize for diagnostic to avoid OOM
max_dim = 1024
w, h = original_image.size
scale = max_dim / max(w, h)
test_image = original_image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

batch = [BatchInputItem(image=test_image, prompt_type="ocr_layout")]

with torch.inference_mode():
    results = generate_hf(batch, model)

print(f"Type of results: {type(results)}")
if len(results) > 0:
    print(f"Type of first result: {type(results[0])}")
    print(f"Available attributes in result: {dir(results[0])}")
    # Also try to print the attribute names specifically
    print(f"Vars of result: {vars(results[0]).keys()}")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Type of results: <class 'list'>
Type of first result: <class 'chandra.model.schema.GenerationResult'>
Available attributes in result: ['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'error', 'raw', 'token_count']
Vars of result: dict_keys(['raw', 'token_count', 'error'])


## Step 2.7 — Run Extraction Across All 52 Pages

In [ ]:
# Reduce max dimension further if OOM persists on large pages
MAX_INFERENCE_DIM = 1400  # down from 1600
print(f"MAX_INFERENCE_DIM set to {MAX_INFERENCE_DIM}")

MAX_INFERENCE_DIM set to 1400


In [ ]:
import os, json, shutil, torch, gc
from pathlib import Path

# ── STEP A: Delete old invalid JSONs from previous failed runs ─────────────
# The old files (from the parse_markdown bug) all have empty blocks.
# The skip-if-exists logic would silently re-use them — wipe them first.
# print("Clearing old extraction JSONs from previous runs...")
# for doc_id in DOCUMENT_REGISTRY:
#     extract_dir = os.path.join(OUTPUT_DIR, "extractions", doc_id)
#     if os.path.exists(extract_dir):
#         shutil.rmtree(extract_dir)
#     os.makedirs(extract_dir, exist_ok=True)
#     print(f"  Cleared: extractions/{doc_id}/")

# ── STEP B: Clear leftover GPU memory before starting ─────────────────────
torch.cuda.empty_cache()
gc.collect()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"\nGPU free before run: {free_gb:.2f} GB\n")

# ── STEP C: Main extraction loop ───────────────────────────────────────────
page_manifest = []
failed_pages  = []

for doc_id, meta in DOCUMENT_REGISTRY.items():
    total_pages = meta["total_pages"]
    print(f"\n{'='*50}")
    print(f"Processing: {doc_id} ({total_pages} pages)")
    print(f"{'='*50}")

    for page_num in range(1, total_pages + 1):
        image_path  = os.path.join(IMAGES_DIR, doc_id, f"page_{page_num:03d}.png")
        output_path = os.path.join(OUTPUT_DIR, "extractions", doc_id, f"page_{page_num:03d}.json")

        # Safe resume: skip pages that were already successfully extracted
        if os.path.exists(output_path):
            with open(output_path, "r", encoding="utf-8") as f:
                saved = json.load(f)
            block_count = len(saved.get("blocks", []))
            print(f"  [SKIP] Page {page_num:03d} — already extracted ({block_count} blocks)")
            page_manifest.append({
                "doc_id":          doc_id,
                "page_num":        page_num,
                "image_path":      image_path,
                "extraction_path": output_path,
                "block_count":     block_count,
                "status":          "skipped",
            })
            continue

        try:
            # flush=True ensures the [RUN] line appears before inference starts
            print(f"  [RUN]  Page {page_num:03d} / {total_pages:03d} ...", end=" ", flush=True)
            page_data   = extract_page(doc_id, page_num, image_path)
            block_count = len(page_data["blocks"])

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(page_data, f, ensure_ascii=False, indent=2)

            print(f"✓ ({block_count} blocks)")
            page_manifest.append({
                "doc_id":          doc_id,
                "page_num":        page_num,
                "image_path":      image_path,
                "extraction_path": output_path,
                "block_count":     block_count,
                "status":          "success",
            })

        except Exception as e:
            print(f"✗ ERROR: {e}")

            # CRITICAL: force GPU cleanup even when extract_page failed mid-way.
            # Without this, the partial allocation from the failed page stays
            # on the GPU and causes cascading OOM on every subsequent page.
            torch.cuda.empty_cache()
            gc.collect()

            failed_pages.append({
                "doc_id":   doc_id,
                "page_num": page_num,
                "error":    str(e),
            })
            page_manifest.append({
                "doc_id":          doc_id,
                "page_num":        page_num,
                "image_path":      image_path,
                "extraction_path": output_path,
                "status":          "failed",
                "error":           str(e),
            })

# ── STEP D: Summary ────────────────────────────────────────────────────────
from collections import Counter
status_counts = Counter(item["status"] for item in page_manifest)

print(f"\n{'='*50}")
print(f"Extraction complete.")
print(f"  Total : {len(page_manifest)} pages")
for status, count in status_counts.items():
    print(f"  {status:10s}: {count}")

if failed_pages:
    print(f"\nFailed pages:")
    for item in failed_pages:
        print(f"  - {item['doc_id']} page {item['page_num']:03d}: {item['error'][:80]}")



GPU free before run: 12.16 GB


Processing: rtgs (19 pages)
  [SKIP] Page 001 — already extracted (7 blocks)
  [SKIP] Page 002 — already extracted (7 blocks)
  [SKIP] Page 003 — already extracted (3 blocks)
  [SKIP] Page 004 — already extracted (12 blocks)
  [SKIP] Page 005 — already extracted (11 blocks)
  [SKIP] Page 006 — already extracted (11 blocks)
  [SKIP] Page 007 — already extracted (24 blocks)
  [SKIP] Page 008 — already extracted (18 blocks)
  [SKIP] Page 009 — already extracted (14 blocks)
  [SKIP] Page 010 — already extracted (16 blocks)
  [SKIP] Page 011 — already extracted (14 blocks)
  [SKIP] Page 012 — already extracted (7 blocks)
  [SKIP] Page 013 — already extracted (7 blocks)
  [SKIP] Page 014 — already extracted (19 blocks)
  [SKIP] Page 015 — already extracted (12 blocks)
  [SKIP] Page 016 — already extracted (13 blocks)
  [SKIP] Page 017 — already extracted (4 blocks)
  [SKIP] Page 018 — already extracted (5 blocks)
  [SKIP] Page 019 — already extracted (2 block

In [ ]:
# DIAGNOSTIC — Run on a single page only, do not save output
import json
from PIL import Image
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown
# Use any small page
test_image_path = f"{IMAGES_DIR}/legal_circular_1/page_001.png"
image = Image.open(test_image_path).convert("RGB")
image_small, _ = resize_for_inference(image, 1024)
batch = [BatchInputItem(image=image_small, prompt_type="ocr_layout")]
with torch.inference_mode():
    result = generate_hf(batch, model)[0]
# ── Inspect the result object ──────────────────────────────────────
print("=== result type ===")
print(type(result))
print("\n=== result attributes (dir) ===")
print([a for a in dir(result) if not a.startswith("_")])
# ── Inspect result.raw ─────────────────────────────────────────────
print("\n=== result.raw (first 1000 chars) ===")
print(result.raw[:1000])
# ── Check parse_markdown return type ──────────────────────────────
parsed = parse_markdown(result.raw)
print("\n=== parse_markdown return type ===")
print(type(parsed))
print("\n=== parse_markdown attributes ===")
print([a for a in dir(parsed) if not a.startswith("_")])
# ── Try iterating to see what we actually get ─────────────────────
items = list(parsed)
print(f"\n=== Items when iterating parse_markdown: {len(items)} items ===")
if items:
    first = items[0]
    print(f"First item type: {type(first)}")
    print(f"First item repr: {repr(first)[:200]}")
    print(f"First item dir: {[a for a in dir(first) if not a.startswith('_')]}")
# ── Check if result has elements/spans/blocks attribute ───────────
for attr in ["elements", "spans", "blocks", "lines", "chunks"]:
    val = getattr(result, attr, "MISSING")
    if val != "MISSING":
        print(f"\n=== result.{attr} ===")
        print(f"  type: {type(val)}")
        print(f"  length: {len(val) if hasattr(val, '__len__') else 'N/A'}")
        if hasattr(val, '__len__') and len(val) > 0:
            first_elem = val[0]
            print(f"  first element type: {type(first_elem)}")
            print(f"  first element dir: {[a for a in dir(first_elem) if not a.startswith('_')]}")
            print(f"  first element repr: {repr(first_elem)[:300]}")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


=== result type ===
<class 'chandra.model.schema.GenerationResult'>

=== result attributes (dir) ===
['error', 'raw', 'token_count']

=== result.raw (first 1000 chars) ===
<div data-bbox="198 145 397 163" data-label="Page-Header"><p>القاهرة في: ٢٢ أبريل ٢٠٢٦</p></div><div data-bbox="773 174 884 192" data-label="Page-Header"><p>السيد الأستاذ/</p></div><div data-bbox="722 203 884 223" data-label="Page-Header"><p>رئيس مجلس الإدارة</p></div><div data-bbox="852 231 884 250" data-label="Page-Header"><p>بنك</p></div><div data-bbox="735 260 867 279" data-label="Page-Header"><p>تحية طيبة وبعد،</p></div><div data-bbox="113 315 886 390" data-label="Text"><p>في إطار حرص البنك المركزي على الحفاظ على استقرار وسلامة القطاع المصرفي، وفي ضوء متابعة التطورات الحالية وبهدف إرساء إطار رقابي فعال للحد من المخاطر المرتبطة بتحويل عمليات شراء الأوراق المالية بالباش. فقد قرر مجلس إدارة البنك المركزي جلسته المنعقدة في ٢١ أبريل ٢٠٢٦ الآتي:</p></div><div data-bbox="245 402 886 423" data-label="Text"><p>"يتعين على

## Step 2.8 — Retry Failed Pages (if any)

In [ ]:
if failed_pages:
    print(f"Retrying {len(failed_pages)} failed pages...\n")
    still_failed = []

    for item in failed_pages:
        doc_id = item["doc_id"]
        page_num = item["page_num"]
        image_filename = f"page_{page_num:03d}.png"
        image_path = os.path.join(IMAGES_DIR, doc_id, image_filename)
        output_path = os.path.join(
            OUTPUT_DIR, "extractions", doc_id, f"page_{page_num:03d}.json"
        )

        try:
            print(f"  Retrying {doc_id} page {page_num:03d} ...", end=" ")
            page_data = extract_page(doc_id, page_num, image_path)
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(page_data, f, ensure_ascii=False, indent=2)
            print("✓")
        except Exception as e:
            print(f"✗ Still failing: {e}")
            still_failed.append(item)

    if still_failed:
        print(f"\n{len(still_failed)} pages could not be processed:")
        for item in still_failed:
            print(f"  - {item['doc_id']} page {item['page_num']:03d}: {item['error']}")
else:
    print("No failed pages — all good!")

## Step 2.9 — Write the Page Manifest

In [ ]:
manifest_path = os.path.join(OUTPUT_DIR, "page_manifest.json")

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(page_manifest, f, ensure_ascii=False, indent=2)

print(f"Page manifest written: {manifest_path}")
print(f"  Entries: {len(page_manifest)}")

# Summary breakdown
from collections import Counter
status_counts = Counter(item["status"] for item in page_manifest)
for status, count in status_counts.items():
    print(f"  {status}: {count}")

Page manifest written: /content/drive/MyDrive/nbe_rag_pipeline/output/page_manifest.json
  Entries: 52
  skipped: 52


## Step 2.10 — Validate Outputs

In [ ]:
import json, os

print("=== Stage 2 Validation ===\n")

all_ok = True

for doc_id, meta in DOCUMENT_REGISTRY.items():
    total_pages = meta["total_pages"]
    extracted_dir = os.path.join(OUTPUT_DIR, "extractions", doc_id)
    found = sorted([f for f in os.listdir(extracted_dir) if f.endswith(".json")])
    print(f"{doc_id}: {len(found)}/{total_pages} pages extracted")

    if len(found) != total_pages:
        print(f"  ⚠️  MISSING PAGES!")
        all_ok = False

    # Spot-check first page
    if found:
        with open(os.path.join(extracted_dir, found[0])) as f:
            sample = json.load(f)
        block_count = len(sample.get("blocks", []))
        has_bbox = any(b.get("bbox") is not None for b in sample["blocks"])
        print(f"  First page: {block_count} blocks, bbox present: {has_bbox}")

print()
if all_ok:
    print("✅ Validation passed — all pages extracted. Ready for Stage 3.")
else:
    print("❌ Validation failed — check missing pages before proceeding.")

=== Stage 2 Validation ===

rtgs: 19/19 pages extracted
  First page: 7 blocks, bbox present: True
legal_circular_1: 2/2 pages extracted
  First page: 11 blocks, bbox present: True
legal_circular_2: 31/31 pages extracted
  First page: 4 blocks, bbox present: True

✅ Validation passed — all pages extracted. Ready for Stage 3.


## Step 3.0 — Confirm Environment Variables Are Set

In [ ]:
import os
from pathlib import Path

BASE_DIR   = "/content/drive/MyDrive/nbe_rag_pipeline"
IMAGES_DIR = os.path.join(BASE_DIR, "docs_images")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

DOCUMENT_REGISTRY = {
    "rtgs": {
        "doc_id": "rtgs",
        "doc_name": "egyptian_multicurrency_rtgs_procedures.pdf",
        "doc_type": "SOP",
        "language": "ar",
        "total_pages": 19,
    },
    "legal_circular_1": {
        "doc_id": "legal_circular_1",
        "doc_name": "legal_circular_1.pdf",
        "doc_type": "Legal Circular",
        "language": "ar",
        "total_pages": 2,
    },
    "legal_circular_2": {
        "doc_id": "legal_circular_2",
        "doc_name": "legal_circular_2.pdf",
        "doc_type": "Legal Circular",
        "language": "ar",
        "total_pages": 31,
    },
}

print("Environment variables set ✓")

Environment variables set ✓


## Step 3.1 — Install / Confirm Pillow

In [ ]:
from PIL import Image
print(f"Pillow available ✓  (version: {Image.__version__})")

Pillow available ✓  (version: 11.3.0)


## Step 3.2 — Create Crop Output Directories

In [ ]:
for doc_id in DOCUMENT_REGISTRY:
    crop_dir = os.path.join(OUTPUT_DIR, "crops", doc_id)
    Path(crop_dir).mkdir(parents=True, exist_ok=True)

print("Crop output directories created ✓")
print("Structure:")
for doc_id in DOCUMENT_REGISTRY:
    print(f"  output/crops/{doc_id}/")

Crop output directories created ✓
Structure:
  output/crops/rtgs/
  output/crops/legal_circular_1/
  output/crops/legal_circular_2/


## Step 3.3 — Define the Crop Function

In [ ]:
from PIL import Image
import os

CROP_PADDING_PX = 8  # pixels of context added around every bounding box

def crop_block(
    page_image: Image.Image,
    block: dict,
    output_path: str,
) -> bool:
    """
    Crop a single block region from the page image and save it.

    Args:
        page_image:  PIL Image of the full page
        block:       Block dict (must contain 'bbox': [x1, y1, x2, y2])
        output_path: Where to save the crop PNG

    Returns:
        True if crop was saved successfully, False if bbox is invalid/missing.
    """
    bbox = block.get("bbox")

    # Skip blocks with no bounding box (e.g. pure metadata blocks)
    if bbox is None or len(bbox) != 4:
        return False

    x1, y1, x2, y2 = bbox
    img_w, img_h = page_image.size

    # Sanity check — coordinates must be positive and within image bounds
    if x2 <= x1 or y2 <= y1:
        return False

    # Apply padding (clamp to image boundaries)
    x1_padded = max(0,     int(x1) - CROP_PADDING_PX)
    y1_padded = max(0,     int(y1) - CROP_PADDING_PX)
    x2_padded = min(img_w, int(x2) + CROP_PADDING_PX)
    y2_padded = min(img_h, int(y2) + CROP_PADDING_PX)

    # Crop and save
    crop = page_image.crop((x1_padded, y1_padded, x2_padded, y2_padded))
    crop.save(output_path, format="PNG", optimize=True)
    return True

## Step 3.4 — Run Crop Generation Across All Documents

In [ ]:
import json
from PIL import Image
from pathlib import Path

crop_manifest = []     # global index: block_id → crop_path
skipped_blocks = []    # blocks with no valid bbox
error_blocks   = []    # blocks that caused exceptions

for doc_id, meta in DOCUMENT_REGISTRY.items():
    total_pages = meta["total_pages"]
    print(f"\n{'='*50}")
    print(f"Generating crops: {doc_id} ({total_pages} pages)")
    print(f"{'='*50}")

    for page_num in range(1, total_pages + 1):
        json_path  = os.path.join(OUTPUT_DIR, "extractions", doc_id, f"page_{page_num:03d}.json")
        image_path = os.path.join(IMAGES_DIR, doc_id, f"page_{page_num:03d}.png")

        # Load extraction JSON
        if not os.path.exists(json_path):
            print(f"  [WARN] Missing extraction JSON for page {page_num:03d} — skipping")
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            page_data = json.load(f)

        # Load page image once per page (reuse across all blocks on this page)
        page_image = Image.open(image_path).convert("RGB")

        crops_on_page = 0
        page_modified = False

        for block in page_data["blocks"]:
            block_id  = block["block_id"]
            crop_filename = f"{block_id}.png"
            crop_path = os.path.join(OUTPUT_DIR, "crops", doc_id, crop_filename)

            # Relative path stored in JSON (portable across systems)
            crop_relative = f"output/crops/{doc_id}/{crop_filename}"

            # Skip if already cropped (safe resume)
            if os.path.exists(crop_path):
                block["crop_path"] = crop_relative
                page_modified = True
                crop_manifest.append({
                    "block_id":   block_id,
                    "doc_id":     doc_id,
                    "page_num":   page_num,
                    "block_type": block.get("block_type", "unknown"),
                    "crop_path":  crop_relative,
                    "status":     "skipped",
                })
                crops_on_page += 1
                continue

            try:
                success = crop_block(page_image, block, crop_path)

                if success:
                    block["crop_path"] = crop_relative
                    page_modified = True
                    crops_on_page += 1
                    crop_manifest.append({
                        "block_id":   block_id,
                        "doc_id":     doc_id,
                        "page_num":   page_num,
                        "block_type": block.get("block_type", "unknown"),
                        "crop_path":  crop_relative,
                        "status":     "success",
                    })
                else:
                    block["crop_path"] = None
                    skipped_blocks.append({
                        "block_id": block_id,
                        "reason":   "no valid bbox",
                    })
                    crop_manifest.append({
                        "block_id":   block_id,
                        "doc_id":     doc_id,
                        "page_num":   page_num,
                        "block_type": block.get("block_type", "unknown"),
                        "crop_path":  None,
                        "status":     "skipped_no_bbox",
                    })

            except Exception as e:
                block["crop_path"] = None
                error_blocks.append({
                    "block_id": block_id,
                    "doc_id":   doc_id,
                    "page_num": page_num,
                    "error":    str(e),
                })
                crop_manifest.append({
                    "block_id":   block_id,
                    "doc_id":     doc_id,
                    "page_num":   page_num,
                    "block_type": block.get("block_type", "unknown"),
                    "crop_path":  None,
                    "status":     "error",
                    "error":      str(e),
                })

        # Write updated JSON back (with crop_path fields filled in)
        if page_modified:
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(page_data, f, ensure_ascii=False, indent=2)

        print(f"  Page {page_num:03d}: {crops_on_page} crops generated")

# Final summary
print(f"\n\nCrop generation complete.")
print(f"  Total manifest entries : {len(crop_manifest)}")
print(f"  Skipped (no bbox)      : {len(skipped_blocks)}")
print(f"  Errors                 : {len(error_blocks)}")


Generating crops: rtgs (19 pages)
  Page 001: 7 crops generated
  Page 002: 7 crops generated
  Page 003: 3 crops generated
  Page 004: 12 crops generated
  Page 005: 11 crops generated
  Page 006: 11 crops generated
  Page 007: 24 crops generated
  Page 008: 18 crops generated
  Page 009: 14 crops generated
  Page 010: 16 crops generated
  Page 011: 14 crops generated
  Page 012: 7 crops generated
  Page 013: 7 crops generated
  Page 014: 19 crops generated
  Page 015: 12 crops generated
  Page 016: 13 crops generated
  Page 017: 4 crops generated
  Page 018: 5 crops generated
  Page 019: 2 crops generated

Generating crops: legal_circular_1 (2 pages)
  Page 001: 11 crops generated
  Page 002: 4 crops generated

Generating crops: legal_circular_2 (31 pages)
  Page 001: 4 crops generated
  Page 002: 6 crops generated
  Page 003: 9 crops generated
  Page 004: 6 crops generated
  Page 005: 8 crops generated
  Page 006: 8 crops generated
  Page 007: 6 crops generated
  Page 008: 7 crops 